**Lab type:** debug  
**Course:** DS104 — Statistics for Data Science  
**Lesson:** Percentiles, Quartiles, and Outlier Detection  
**Task:** The outlier detection pipeline below contains 3 bugs. Each runs without errors but produces incorrect results — either wrong thresholds, excessive removal, or blind removal of potentially meaningful data points. Find each bug, explain what it does wrong, and fix it.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(99)

# Simulated logistics dataset: delivery times in minutes
# Contains some legitimate long deliveries (delays due to weather, traffic)
n = 1200
normal_deliveries = np.random.normal(loc=45, scale=10, size=int(n * 0.92))
delayed_deliveries = np.random.normal(loc=120, scale=15, size=int(n * 0.08))  # real delays
delivery_times = np.concatenate([normal_deliveries, delayed_deliveries])
delivery_times = np.clip(delivery_times, 5, None)  # no negative times
np.random.shuffle(delivery_times)

df = pd.DataFrame({'delivery_minutes': delivery_times})

print(f'Dataset: {len(df)} deliveries')
print(f'Mean: {df.delivery_minutes.mean():.1f} min')
print(f'Median: {df.delivery_minutes.median():.1f} min')
print(f'Max: {df.delivery_minutes.max():.1f} min')

## Bug 1: IQR rule applied with the wrong boundary

The code below attempts to flag outliers using the 1.5×IQR rule, but the lower bound is calculated incorrectly.

In [ ]:
# --- BUGGY CODE ---
Q1 = df.delivery_minutes.quantile(0.25)
Q3 = df.delivery_minutes.quantile(0.75)
IQR = Q3 - Q1

# BUG: lower bound uses Q1 alone, not Q1 - 1.5*IQR
lower_bound = Q1
upper_bound = Q3 + 1.5 * IQR

outliers_buggy = df[
    (df.delivery_minutes < lower_bound) | (df.delivery_minutes > upper_bound)
]

print(f'Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}')
print(f'Lower bound: {lower_bound:.1f} (correct would be {Q1 - 1.5*IQR:.1f})')
print(f'Upper bound: {upper_bound:.1f}')
print(f'Outliers flagged: {len(outliers_buggy)} ({len(outliers_buggy)/len(df)*100:.1f}% of data)')

**Explanation:** Write your answer here — what does using `Q1` as the lower bound instead of `Q1 - 1.5*IQR` cause? How many rows are incorrectly flagged?

*(Write your answer here.)*

**Fix the bug:**

In [ ]:
# Corrected IQR rule
Q1 = df.delivery_minutes.quantile(0.25)
Q3 = df.delivery_minutes.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_fixed = df[
    (df.delivery_minutes < lower_bound) | (df.delivery_minutes > upper_bound)
]

print(f'Lower bound: {lower_bound:.1f}, Upper bound: {upper_bound:.1f}')
print(f'Outliers flagged: {len(outliers_fixed)} ({len(outliers_fixed)/len(df)*100:.1f}% of data)')

## Bug 2: Z-score threshold too aggressive

The code uses z-score outlier detection with a threshold of 1.5. This is far too aggressive for most real datasets.

In [ ]:
# --- BUGGY CODE ---
z_scores = np.abs(stats.zscore(df.delivery_minutes))

# BUG: threshold of 1.5 flags ~13% of any normal distribution as outliers
threshold = 1.5
outliers_zscore = df[z_scores > threshold]

print(f'Z-score threshold: {threshold}')
print(f'Rows flagged as outliers: {len(outliers_zscore)} ({len(outliers_zscore)/len(df)*100:.1f}%)')
print(f'Min delivery time in flagged rows: {outliers_zscore.delivery_minutes.min():.1f}')
print(f'Max delivery time in flagged rows: {outliers_zscore.delivery_minutes.max():.1f}')

**Explanation:** Write your answer here — what percentage of a perfectly normal distribution falls outside z=±1.5? Why does this make it a bad default threshold for outlier detection?

*(Write your answer here.)*

**Fix the bug:**

In [ ]:
# Corrected z-score threshold
z_scores = np.abs(stats.zscore(df.delivery_minutes))

# Standard threshold: 3 (flags ~0.3% of a normal distribution)
threshold = 3.0
outliers_zscore_fixed = df[z_scores > threshold]

print(f'Z-score threshold: {threshold}')
print(f'Rows flagged as outliers: {len(outliers_zscore_fixed)} ({len(outliers_zscore_fixed)/len(df)*100:.1f}%)')
print(f'Min delivery time in flagged rows: {outliers_zscore_fixed.delivery_minutes.min():.1f}')
print(f'Max delivery time in flagged rows: {outliers_zscore_fixed.delivery_minutes.max():.1f}')

## Bug 3: Outliers removed without domain investigation

The dataset contains delivery times from 120–160 minutes — these are the simulated *delayed deliveries* caused by weather and traffic. The code below removes them automatically.

In [ ]:
# --- BUGGY CODE ---
Q1 = df.delivery_minutes.quantile(0.25)
Q3 = df.delivery_minutes.quantile(0.75)
IQR = Q3 - Q1

# This uses the correct IQR formula, but immediately drops all flagged rows
# without any investigation
df_clean = df[
    (df.delivery_minutes >= Q1 - 1.5 * IQR) &
    (df.delivery_minutes <= Q3 + 1.5 * IQR)
].copy()

print(f'Rows before: {len(df)}')
print(f'Rows after removal: {len(df_clean)}')
print(f'Rows removed: {len(df) - len(df_clean)}')
print(f'Max delivery in cleaned data: {df_clean.delivery_minutes.max():.1f} min')

**Explanation:** Write your answer here — why is automatic outlier removal wrong here? What was lost from the dataset, and why does that matter for the analysis?

*(Write your answer here.)*

**Better approach:**

In [ ]:
# Better: flag outliers for investigation, don't drop blindly
Q1 = df.delivery_minutes.quantile(0.25)
Q3 = df.delivery_minutes.quantile(0.75)
IQR = Q3 - Q1

df['is_outlier'] = (
    (df.delivery_minutes < Q1 - 1.5 * IQR) |
    (df.delivery_minutes > Q3 + 1.5 * IQR)
)

print('Outlier summary:')
print(df.groupby('is_outlier').delivery_minutes.agg(['count', 'min', 'max', 'mean']).round(1))
print()
print('Decision: these long deliveries are real delay events, not data errors.')
print('Keep them but analyse the two populations separately.')

## Summary

> **Final question:** In one sentence each, state the three lessons from this lab — one for each bug.

*(Write your three lessons here.)*

1. 
2. 
3. 